# Nanochat Universal RL Run on Kaggle

This notebook starts from the saved `d8_kaggle` SFT checkpoint and runs `scripts.chat_universal_rl` with three objectives:

- `reinforce`
- `rloo_kl`
- `gspo`

Recommended Kaggle setup:

- GPU enabled (`2x Tesla T4` expected)
- Internet enabled
- Attach `nanochat-codes`
- Attach `nanochat-d8-sft-cache`

The notebook imports only the tokenizer and `chatsft_checkpoints/d8_kaggle` from the SFT cache, then writes objective-specific checkpoints under `chatrl_checkpoints`.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path('/kaggle/input')

CODE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-codes'))
SFT_CACHE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-d8-sft-cache'))

assert CODE_INPUTS, 'Attach the nanochat-codes Kaggle dataset'
assert SFT_CACHE_INPUTS, 'Attach the nanochat-d8-sft-cache Kaggle dataset'

CODE_INPUT = CODE_INPUTS[0]
SFT_CACHE_INPUT = SFT_CACHE_INPUTS[0]

WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

required_cache_paths = [
    Path('tokenizer'),
    Path('chatsft_checkpoints') / 'd8_kaggle',
]
for rel_path in required_cache_paths:
    src_path = SFT_CACHE_INPUT / rel_path
    dst_path = WORK_CACHE / rel_path
    assert src_path.exists(), f'Missing required cache path in attached dataset: {src_path}'
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')

print('Code input:', CODE_INPUT)
print('SFT cache input:', SFT_CACHE_INPUT)
print('Working repo:', WORK_REPO)
print('Nanochat cache:', WORK_CACHE)
print('HF cache:', HF_CACHE)
subprocess.run(['df', '-h', '/kaggle/working'], check=False)
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)


Code input: /kaggle/input/datasets/jennyqqjiang/nanochat-codes
SFT cache input: /kaggle/input/datasets/yshuaiqin/nanochat-d8-sft-cache
Working repo: /kaggle/working/nanochat
Nanochat cache: /kaggle/working/nanochat_cache
HF cache: /kaggle/working/huggingface_cache
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.1G   18G  11% /kaggle/working
2.1G	/kaggle/working/nanochat_cache


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Install Dependencies

Install the runtime packages the Kaggle image may not already include.


In [2]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'requests>=2.32.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
    'rustbpe>=0.1.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.7 MB/s eta 0:00:00
Dependencies installed


## Configure Runtime

Universal RL follows the same runtime settings as the RL notebook and keeps compile disabled for Kaggle stability.


In [3]:
env = os.environ.copy()
env['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
env['HF_HOME'] = str(HF_CACHE)
env['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
env['PYTHONUNBUFFERED'] = '1'

env['NANOCHAT_DTYPE'] = 'float16'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['TORCH_COMPILE_DISABLE'] = '1'
env['NANOCHAT_ADAMW_ALLREDUCE'] = '1'
env['NANOCHAT_SERIAL_OPTIM_COMM'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'
env['TORCH_NCCL_ASYNC_ERROR_HANDLING'] = '1'
env['NCCL_DEBUG'] = 'WARN'

os.environ.update(env)

import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## Validate Repo And SFT Cache

Check the active Kaggle working copy and attached SFT cache before starting training.


In [4]:
assert (WORK_REPO / 'scripts' / 'chat_universal_rl.py').exists(), 'Missing scripts/chat_universal_rl.py'
assert (WORK_REPO / 'scripts' / 'chat_post_eval.py').exists(), 'Missing scripts/chat_post_eval.py'
assert (WORK_REPO / 'scripts' / 'chat_web.py').exists(), 'Missing scripts/chat_web.py'
assert (WORK_REPO / 'nanochat' / 'checkpoint_manager.py').exists(), 'Missing nanochat/checkpoint_manager.py'

MODEL_TAG = 'd8_kaggle'
sft_dir = WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG
tokenizer_dir = WORK_CACHE / 'tokenizer'
assert sft_dir.exists(), f'Missing SFT checkpoint dir: {sft_dir}'
assert tokenizer_dir.exists(), f'Missing tokenizer dir: {tokenizer_dir}'
assert sorted(sft_dir.glob('model_*.pt')), f'No model_*.pt files found in {sft_dir}'
assert sorted(sft_dir.glob('meta_*.json')), f'No meta_*.json files found in {sft_dir}'

subprocess.check_call(
    [
        sys.executable, '-m', 'py_compile',
        'scripts/chat_universal_rl.py',
        'scripts/chat_post_eval.py',
        'scripts/chat_web.py',
        'nanochat/checkpoint_manager.py',
    ],
    cwd=WORK_REPO,
    env=env,
)

print('Setup complete')
print('SFT checkpoints:', [p.name for p in sorted(sft_dir.glob('model_*.pt'))[-5:]])
print('Tokenizer files:', sorted(p.name for p in tokenizer_dir.iterdir()))


Setup complete
SFT checkpoints: ['model_004999.pt']
Tokenizer files: ['token_bytes.pt', 'tokenizer.pkl']


## Universal RL Settings

These settings run each objective long enough to produce useful checkpoints while keeping the full three-objective notebook practical on Kaggle.


In [5]:
NPROC = 2 if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else 1
MODEL_TAG = 'd8_kaggle'
UNIVERSAL_OBJECTIVES = ['reinforce', 'rloo_kl', 'gspo']

MAX_STEPS = 100
DEVICE_BATCH_SIZE = 2
EXAMPLES_PER_STEP = 8
NUM_SAMPLES = 16
MAX_NEW_TOKENS = 256
EVAL_EVERY = 25
EVAL_EXAMPLES = 100
SAVE_EVERY = 50

def objective_config(objective):
    if objective == 'reinforce':
        return {
            'kl_beta': 0.0,
            'update_epochs': 1,
            'extra_args': [],
            'embedding_lr': '0.03',
            'unembedding_lr': '0.0008',
            'matrix_lr': '0.002',
        }
    if objective == 'rloo_kl':
        return {
            'kl_beta': 0.02,
            'update_epochs': 1,
            'extra_args': [],
            'embedding_lr': '0.03',
            'unembedding_lr': '0.0008',
            'matrix_lr': '0.002',
        }
    if objective == 'gspo':
        return {
            'kl_beta': 0.0,
            'update_epochs': 2,
            'extra_args': ['--clip-epsilon=0.2'],
            'embedding_lr': '0.02',
            'unembedding_lr': '0.0006',
            'matrix_lr': '0.0015',
        }
    raise ValueError(f'Unknown objective: {objective}')

def run_universal_rl(objective, overwrite=True):
    cfg = objective_config(objective)
    output_tag = f'{MODEL_TAG}_{objective}'
    output_dir = WORK_CACHE / 'chatrl_checkpoints' / output_tag

    if overwrite:
        shutil.rmtree(output_dir, ignore_errors=True)

    rl_args = [
        '-m', 'scripts.chat_universal_rl',

        '--run=dummy',
        '--model-source=sft',
        f'--model-tag={MODEL_TAG}',

        '--reference-source=sft',
        f'--reference-tag={MODEL_TAG}',

        '--objective', objective,
        '--kl-beta', str(cfg['kl_beta']),
        '--update-epochs', str(cfg['update_epochs']),

        f'--max-steps={MAX_STEPS}',
        f'--device-batch-size={DEVICE_BATCH_SIZE}',
        f'--examples-per-step={EXAMPLES_PER_STEP}',
        f'--num-samples={NUM_SAMPLES}',
        f'--max-new-tokens={MAX_NEW_TOKENS}',

        '--temperature=1.0',
        '--top-k=0',

        f'--eval-every={EVAL_EVERY}',
        f'--eval-examples={EVAL_EXAMPLES}',
        f'--save-every={SAVE_EVERY}',

        f"--embedding-lr={cfg['embedding_lr']}",
        f"--unembedding-lr={cfg['unembedding_lr']}",
        f"--matrix-lr={cfg['matrix_lr']}",
        '--init-lr-frac=0.05',
    ] + cfg['extra_args']

    if NPROC > 1:
        cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + rl_args
    else:
        cmd = [sys.executable] + rl_args

    print('Running command:')
    print(' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)

    print('Output checkpoint dir:', output_dir)
    if output_dir.exists():
        print('Saved checkpoints:', [p.name for p in sorted(output_dir.glob('model_*.pt'))])
    return output_dir


## REINFORCE Run

Run the grouped REINFORCE objective from the SFT checkpoint.


In [6]:
reinforce_dir = run_universal_rl('reinforce')


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_universal_rl --run=dummy --model-source=sft --model-tag=d8_kaggle --reference-source=sft --reference-tag=d8_kaggle --objective reinforce --kl-beta 0.0 --update-epochs 1 --max-steps=100 --device-batch-size=2 --examples-per-step=8 --num-samples=16 --max-new-tokens=256 --temperature=1.0 --top-k=0 --eval-every=25 --eval-examples=100 --save-every=50 --embedding-lr=0.03 --unembedding-lr=0.0008 --matrix-lr=0.002 --init-lr-frac=0.05


[W614 06:30:55.129916889 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 06:31:07,801 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 06:31:07,801 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 06:31:07,803 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 06:31:07,803 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 06:31:09.842248904 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 06:31:09.842643177 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Calculated number of steps: 100
Scaling the LR for the AdamW parameters ∝1/√(512/768) = 1.224745
Total sequences per step: 128
Calculated examples per rank: 4
Step 0 | Pass@1: 0.0000, Pass@2: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 1 | Epoch 0 | loss_contrib: -0.00000

2026-06-14 07:36:48,677 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce/model_000050.pt
2026-06-14 07:36:48,678 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce/meta_000050.json


Saved model checkpoint to /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 1 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 1 | Epoch 0 | loss_contrib: -0.00000

2026-06-14 08:41:51,078 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce/model_000099.pt
2026-06-14 08:41:51,078 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce/meta_000099.json


Saved model checkpoint to /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce
Output checkpoint dir: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce
Saved checkpoints: ['model_000050.pt', 'model_000099.pt']


## RLOO-KL Run

Run leave-one-out REINFORCE with a KL penalty to the SFT reference.


In [7]:
rloo_kl_dir = run_universal_rl('rloo_kl')


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_universal_rl --run=dummy --model-source=sft --model-tag=d8_kaggle --reference-source=sft --reference-tag=d8_kaggle --objective rloo_kl --kl-beta 0.02 --update-epochs 1 --max-steps=100 --device-batch-size=2 --examples-per-step=8 --num-samples=16 --max-new-tokens=256 --temperature=1.0 --top-k=0 --eval-every=25 --eval-examples=100 --save-every=50 --embedding-lr=0.03 --unembedding-lr=0.0008 --matrix-lr=0.002 --init-lr-frac=0.05


[W614 08:41:56.922263664 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 08:42:01,664 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 08:42:01,665 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 08:42:01,668 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 08:42:01,670 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 08:42:02.760635296 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 08:42:02.767889702 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Calculated number of steps: 100
Scaling the LR for the AdamW parameters ∝1/√(512/768) = 1.224745
Total sequences per step: 128
Calculated examples per rank: 4
Step 0 | Pass@1: 0.0000, Pass@2: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss_contrib: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 1 | Epoch 0 | loss_contrib: -0.00000

2026-06-14 09:45:03,361 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl/model_000050.pt
2026-06-14 09:45:03,361 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl/meta_000050.json


Saved model checkpoint to /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: 0.000003 | reward: 0.0000 | return: -0.0028
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000024 | reward: 0.0000 | return: -0.0028
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000084 | reward: 0.0000 | return: -0.0028
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000008 | reward: 0.0000 | return: -0.0028
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: 0.000007 | reward: 0.0000 | return: -0.0028
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: 0.000034 | reward: 0.0000 | return: -0.0028
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: -0.000017 | reward: 0.0000 | return: -0.0028
Step 51/100 | Example 0 | Epoch 0 | loss_contrib: 0.000011 | reward: 0.0000 | return: -0.0028
Step 51/100 | Example 1 | Epoch 0 | loss_contrib: -0.000033 | reward: 0.0000 | return: -0.0036
Step 51/100 | Example 1 | Epoch 0 | loss_contrib: 0.000

2026-06-14 10:48:25,142 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl/model_000099.pt
2026-06-14 10:48:25,143 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl/meta_000099.json


Saved model checkpoint to /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl
Output checkpoint dir: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl
Saved checkpoints: ['model_000050.pt', 'model_000099.pt']


## GSPO Run

Run sequence-level clipped policy optimization with RLOO advantages.


In [8]:
gspo_dir = run_universal_rl('gspo')


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_universal_rl --run=dummy --model-source=sft --model-tag=d8_kaggle --reference-source=sft --reference-tag=d8_kaggle --objective gspo --kl-beta 0.0 --update-epochs 2 --max-steps=100 --device-batch-size=2 --examples-per-step=8 --num-samples=16 --max-new-tokens=256 --temperature=1.0 --top-k=0 --eval-every=25 --eval-examples=100 --save-every=50 --embedding-lr=0.02 --unembedding-lr=0.0006 --matrix-lr=0.0015 --init-lr-frac=0.05 --clip-epsilon=0.2


[W614 10:48:29.590696637 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 10:48:34,279 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 10:48:34,280 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 10:48:34,290 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 10:48:34,292 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 10:48:34.397423477 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 10:48:34.397819744 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Calculated number of steps: 100
Scaling the LR for the AdamW parameters ∝1/√(512/768) = 1.224745
Total sequences per step: 128
Calculated examples per rank: 4
Step 0 | Pass@1: 0.0000, Pass@2: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 1 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 0/100 | Example 1 | Epoch 0 | l

2026-06-14 11:55:21,784 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_gspo/model_000050.pt
2026-06-14 11:55:21,784 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_gspo/meta_000050.json


Saved model checkpoint to /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_gspo
Step 51/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 0 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 1 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 1 | Epoch 0 | loss: -0.000000 | reward: 0.0000 | return: 0.0000
Step 51/100 | Example 1 | Epoch 0 | loss: -0.0000

2026-06-14 12:55:59,481 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_gspo/model_000099.pt
2026-06-14 12:55:59,482 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_gspo/meta_000099.json


Saved model checkpoint to /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_gspo
Output checkpoint dir: /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_gspo
Saved checkpoints: ['model_000050.pt', 'model_000099.pt']


## Inspect Universal RL Checkpoints

Confirm each objective produced checkpoint files before evaluation or saving.


In [9]:
for objective in UNIVERSAL_OBJECTIVES:
    output_tag = f'{MODEL_TAG}_{objective}'
    checkpoint_dir = WORK_CACHE / 'chatrl_checkpoints' / output_tag
    print()
    print(objective, checkpoint_dir)
    print('Exists:', checkpoint_dir.exists())
    if checkpoint_dir.exists():
        print('Files:', [p.name for p in sorted(checkpoint_dir.glob('*'))])

subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)



reinforce /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce
Exists: True
Files: ['meta_000050.json', 'meta_000099.json', 'model_000050.pt', 'model_000099.pt']

rloo_kl /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl
Exists: True
Files: ['meta_000050.json', 'meta_000099.json', 'model_000050.pt', 'model_000099.pt']

gspo /kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_gspo
Exists: True
Files: ['meta_000050.json', 'meta_000099.json', 'model_000050.pt', 'model_000099.pt']
4.9G	/kaggle/working/nanochat_cache


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Comparison Eval

Compare SFT against every Universal RL objective checkpoint on a small fixed evaluation slice.


In [10]:
RUN_COMPARISON_EVAL = True
EVAL_MAX_PROBLEMS = 100

if RUN_COMPARISON_EVAL:
    models = ['sft=sft:d8_kaggle']

    rl_dir = WORK_CACHE / 'chatrl_checkpoints'
    for objective in UNIVERSAL_OBJECTIVES:
        output_tag = f'{MODEL_TAG}_{objective}'
        if (rl_dir / output_tag).exists():
            models.append(f'{objective}=@{rl_dir}:{output_tag}')

    post_eval_args = [
        '-m', 'scripts.chat_post_eval',
    ]
    for model in models:
        post_eval_args.extend(['--models', model])
    post_eval_args.extend([
        '--max-problems', str(EVAL_MAX_PROBLEMS),
        '--batch-size', '2',
    ])

    if NPROC > 1:
        cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + post_eval_args
    else:
        cmd = [sys.executable] + post_eval_args

    print('Running command:')
    print(' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)
else:
    print('Skipping comparison eval')


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_post_eval --models sft=sft:d8_kaggle --models reinforce=@/kaggle/working/nanochat_cache/chatrl_checkpoints:d8_kaggle_reinforce --models rloo_kl=@/kaggle/working/nanochat_cache/chatrl_checkpoints:d8_kaggle_rloo_kl --models gspo=@/kaggle/working/nanochat_cache/chatrl_checkpoints:d8_kaggle_gspo --max-problems 100 --batch-size 2


[W614 12:56:04.688692240 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 12:56:08,424 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 12:56:08,426 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 12:56:08,429 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 12:56:08,430 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 12:56:09.850258271 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 12:56:09.850445078 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Final: 20/100 (20.00%)
gspo | MMLU: 20.00%


2026-06-14 13:26:24,206 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 13:26:24,210 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 13:26:24,222 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 13:26:24,226 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 13:26:24,270 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 13:26:24,271 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k

Rank 0 | 0/50 (0.00%)
Rank 1 | 0/50 (0.00%)
Final: 0/100 (0.00%)
gspo | GSM8K: 0.00%


2026-06-14 13:29:12,280 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 13:29:12,281 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 13:29:12,297 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 13:29:12,297 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 13:29:12,344 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/openai_humaneval.py "HTTP/1.1 404 Not Found"
2026-06-14 13:29:12,346 - httpx - INFO

Rank 0 | 0/50 (0.00%)
Rank 1 | 0/50 (0.00%)
Final: 0/100 (0.00%)
gspo | HumanEval: 0.00%
Rank 0 | 4/50 (8.00%)
Rank 1 | 9/50 (18.00%)
Final: 13/100 (13.00%)
gspo | SpellingBee: 13.00%

Model     | ARC-Easy | ARC-Challenge | MMLU   | GSM8K | HumanEval | SpellingBee | Mean  
----------+----------+---------------+--------+-------+-----------+-------------+-------
sft       | 26.00%   | 29.00%        | 24.00% | 0.00% | 0.00%     | 95.00%      | 29.00%
reinforce | 23.00%   | 31.00%        | 20.00% | 0.00% | 0.00%     | 0.00%       | 12.33%
rloo_kl   | 23.00%   | 32.00%        | 20.00% | 0.00% | 0.00%     | 0.00%       | 12.50%
gspo      | 23.00%   | 32.00%        | 20.00% | 0.00% | 0.00%     | 13.00%      | 14.67%

Ranking by mean score:
1. sft: 29.00%
2. gspo: 14.67%
3. rloo_kl: 12.50%
4. reinforce: 12.33%


## Inspect Saved Comparison Report

Display the report written by `scripts.chat_post_eval`.


In [11]:
report_path = WORK_CACHE / 'report' / 'chat-post-eval.md'
print(report_path)
print('Exists:', report_path.exists())
if report_path.exists():
    print(report_path.read_text())


/kaggle/working/nanochat_cache/report/chat-post-eval.md
Exists: True
## Chat Post Eval
timestamp: 2026-06-14 13:35:38

- tasks: ['ARC-Easy', 'ARC-Challenge', 'MMLU', 'GSM8K', 'HumanEval', 'SpellingBee']
- num_samples: 1
- temperature: 0.0000
- max_new_tokens: 512
- batch_size: 2
- max_problems: 100
- models: [{'label': 'sft', 'origin': 'sft', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 4999, 'resolved_meta_keys': ['model_config', 'step', 'user_config', 'val_bpb']}, {'label': 'reinforce', 'origin': '/kaggle/working/nanochat_cache/chatrl_checkpoints', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce', 'model_tag': 'd8_kaggle_reinforce', 'step': 99, 'resolved_meta_keys': ['model_config']}, {'label': 'rloo_kl', 'origin': '/kaggle/working/nanochat_cache/chatrl_checkpoints', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl', 'model_tag': 'd8_ka

## Output Manifest

Write a compact manifest of the artifacts produced by this notebook.


In [12]:
manifest = {
    'model_tag': MODEL_TAG,
    'objectives': UNIVERSAL_OBJECTIVES,
    'sft_checkpoint_dir': str(WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG),
    'universal_rl_checkpoint_dirs': {
        objective: str(WORK_CACHE / 'chatrl_checkpoints' / f'{MODEL_TAG}_{objective}')
        for objective in UNIVERSAL_OBJECTIVES
    },
    'report': str(WORK_CACHE / 'report' / 'chat-post-eval.md'),
}
manifest_path = Path('/kaggle/working/nanochat_universal_rl_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(manifest_path)
print(manifest_path.read_text())


/kaggle/working/nanochat_universal_rl_manifest.json
{
  "model_tag": "d8_kaggle",
  "objectives": [
    "reinforce",
    "rloo_kl",
    "gspo"
  ],
  "sft_checkpoint_dir": "/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle",
  "universal_rl_checkpoint_dirs": {
    "reinforce": "/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_reinforce",
    "rloo_kl": "/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_rloo_kl",
    "gspo": "/kaggle/working/nanochat_cache/chatrl_checkpoints/d8_kaggle_gspo"
  },
  "report": "/kaggle/working/nanochat_cache/report/chat-post-eval.md"
}


In [13]:
# Optional: save the Universal RL checkpoint cache as a Kaggle Dataset.
import json

UNIVERSAL_RL_CACHE_DIR = Path('/kaggle/working/nanochat_d8_universal_rl_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-universal-rl-cache'
TITLE = 'nanochat d8 universal rl cache'
VERSION_MESSAGE = f'Save {MODEL_TAG} Universal RL checkpoint cache'
UPLOAD_ACTION = 'create'  # use 'version' after the Dataset already exists

required_dirs = [
    WORK_CACHE / 'chatrl_checkpoints' / f'{MODEL_TAG}_{objective}'
    for objective in UNIVERSAL_OBJECTIVES
]
required_dirs.append(WORK_CACHE / 'tokenizer')

for required_dir in required_dirs:
    assert required_dir.exists(), f'Missing required directory: {required_dir}'

for objective in UNIVERSAL_OBJECTIVES:
    objective_dir = WORK_CACHE / 'chatrl_checkpoints' / f'{MODEL_TAG}_{objective}'
    assert sorted(objective_dir.glob('model_*.pt')), f'No checkpoints found for {objective}: {objective_dir}'

if UNIVERSAL_RL_CACHE_DIR.exists():
    shutil.rmtree(UNIVERSAL_RL_CACHE_DIR)
UNIVERSAL_RL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(WORK_CACHE / 'chatrl_checkpoints', UNIVERSAL_RL_CACHE_DIR / 'chatrl_checkpoints')
shutil.copytree(WORK_CACHE / 'tokenizer', UNIVERSAL_RL_CACHE_DIR / 'tokenizer')

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = UNIVERSAL_RL_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(UNIVERSAL_RL_CACHE_DIR)], check=False)

if UPLOAD_ACTION == 'create':
    cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(UNIVERSAL_RL_CACHE_DIR),
        '--dir-mode', 'tar',
    ]
elif UPLOAD_ACTION == 'version':
    cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(UNIVERSAL_RL_CACHE_DIR),
        '-m', VERSION_MESSAGE,
        '--dir-mode', 'tar',
    ]
else:
    raise ValueError("UPLOAD_ACTION must be 'create' or 'version'")

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


Folder size:
2.9G	/kaggle/working/nanochat_d8_universal_rl_cache
Running: kaggle datasets create -p /kaggle/working/nanochat_d8_universal_rl_cache --dir-mode tar
Starting upload for file tokenizer.tar


100%|██████████| 540k/540k [00:00<00:00, 2.12MB/s]


Upload successful: tokenizer.tar (540KB)
Starting upload for file chatrl_checkpoints.tar


100%|██████████| 2.81G/2.81G [00:31<00:00, 97.3MB/s]


Upload successful: chatrl_checkpoints.tar (3GB)
Dataset creation error: Dataset url's dataset slugs and hashlink are all null
Done.
Dataset URL: https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-universal-rl-cache


## Serve the Universal RL Chat Model

Start a local NanoChat server from one Universal RL objective checkpoint.


In [ ]:
import time
import requests

SERVE_OBJECTIVE = 'rloo_kl'  # choose 'reinforce', 'rloo_kl', or 'gspo'
SERVE_MODEL_TAG = f'{MODEL_TAG}_{SERVE_OBJECTIVE}'
SERVER_PORT = 8000
BASE_URL = f'http://127.0.0.1:{SERVER_PORT}'
CHECKPOINT_DIR = WORK_CACHE / 'chatrl_checkpoints'

assert SERVE_OBJECTIVE in UNIVERSAL_OBJECTIVES, f'Unknown objective: {SERVE_OBJECTIVE}'
model_dir = CHECKPOINT_DIR / SERVE_MODEL_TAG
assert model_dir.exists(), f'Missing Universal RL checkpoint directory: {model_dir}'
assert sorted(model_dir.glob('model_*.pt')), f'No model_*.pt files found in {model_dir}'
assert sorted(model_dir.glob('meta_*.json')), f'No meta_*.json files found in {model_dir}'

try:
    if server.poll() is None:
        print('Stopping existing NanoChat server...')
        server.terminate()
        server.wait(timeout=10)
        print('Stopped existing server.')
except NameError:
    pass
except Exception as exc:
    print('Could not stop existing server cleanly:', exc)
    try:
        server.kill()
        server.wait(timeout=10)
        print('Killed existing server.')
    except Exception:
        pass

messages = []

server_env = env.copy()
server_env['NANOCHAT_DISABLE_COMPILE'] = '1'
server_env['TORCH_COMPILE_DISABLE'] = '1'
server_env['OMP_NUM_THREADS'] = '1'

cmd = [
    sys.executable,
    '-m', 'scripts.chat_web',
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--model-tag', SERVE_MODEL_TAG,
    '--num-gpus', '1',
    '--port', str(SERVER_PORT),
]

print('Starting NanoChat server:')
print(' '.join(cmd))
server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f'Server process started with PID {server.pid}.')

SERVER_READY = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f'NanoChat server exited early with code {server.returncode}')
    try:
        response = requests.get(f'{BASE_URL}/health', timeout=2)
        if response.ok:
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f'NanoChat server is ready: {BASE_URL}')
else:
    print(f'NanoChat server is still loading. Wait a bit, then use: {BASE_URL}')


In [ ]:
import json
import requests

BASE = globals().get('BASE_URL', 'http://127.0.0.1:8000')
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({'role': 'user', 'content': prompt})

    payload = {
        'messages': messages,
        'temperature': temperature,
        'top_k': top_k,
        'max_tokens': max_tokens,
    }

    print('Assistant:', end=' ', flush=True)
    answer = ''

    with requests.post(f'{BASE}/chat/completions', json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue

            data = json.loads(line[len('data: '):])
            if data.get('done'):
                break

            token = data.get('token', '')
            answer += token
            print(token, end='', flush=True)

    print()
    messages.append({'role': 'assistant', 'content': answer})
    return answer

print(f'Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.')
while True:
    prompt = input('\nYou: ')
    command = prompt.strip().lower()
    if command in {'q', 'quit', 'exit'}:
        break
    if command == 'reset':
        messages.clear()
        print('Chat history cleared.')
        continue
    ask(prompt)


In [ ]:
# Stop the NanoChat web server started by the serving cell.
import os
import subprocess
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat server killed.')
        stopped_any = True
    else:
        print(f'NanoChat server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `server` variable found.')

try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat server process found.')
